# TAE-IA · Module 6 · L25 — «¿Qué suena aquí?» · **the vision half, and the join**

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Session** | L25 — Track C, opening |
| **What this is** | The second half of Friday's skeleton. At the end of it you have **one** app. |
| **Due** | Today in class, or as homework with no penalty |
| **Graded** | No. What counts is that it runs and that you show it |

---

## What you are building today

On Friday your app learned to **listen**: CLAP tagged a clip against a vocabulary you wrote.
Today it learns to **look**, with the same vocabulary — and then it compares the two.

```
   audio clip                         photo
       |                                |
 [ coerce_audio ]                [ coerce_image ]     ← both given
       |                                |
 [ CLAP + Whisper ]                 [ CLIP ]
  your L24 half                    today's half
       |                                |
 analyse_audio(...)              analyse_image(...)
       |                                |
       +----------- same SCENES --------+
                        |
                  [ join(a, v) ]   ← do they agree?
                        |
                    [ Gradio ]
```

**CLIP is to images what CLAP is to audio**: it turns the photo into a vector, turns each of
your phrases into a vector in the same space, and answers with the closest. That is why one
vocabulary works for both — and why the join is short.

---

## The spec

Your app must, **at minimum**:

1. Contain your **L24 audio half**, working (section 2).
2. Tag an image with **CLIP** against the **same `SCENES`** your audio half uses.
3. **Refuse to guess** on an image below a floor you measured — its own floor, not CLAP's.
4. `analyse_image` returns **the same five keys** as `analyse_audio`.
5. `join` says whether the sound and the image agree — and says so honestly when either side
   was not sure enough to compare.
6. One Gradio app: `gr.Audio` **and** `gr.Image` in, a scene dropdown, one button, one report.

And it must **pass the test cell** in section 7.

## The models

| For | Model | Have you used it? |
|---|---|---|
| Tagging sound | `laion/clap-htsat-unfused` | Yes — Friday |
| Detecting speech | Whisper `small` | Yes — L19, L20, Friday |
| Tagging images | `openai/clip-vit-base-patch32` | Yes — L12's router. **New today: the `pipeline` way to call it** |

## Rules

Same as Friday. Use whatever you like, including an assistant. The test cell decides, not effort.
Anything unfinished goes home as homework with no penalty. At the end, a few of you show what
runs — finished or not.

---

## 1 — Install and setup · **given, identical to Friday**

In [1]:
# Two packages. CLAP rides inside `transformers`, which Colab already has.
!pip install -q openai-whisper librosa

In [2]:
import os, sys, gc, time, random
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/TAE_IA_M6'
OUTPUT_DIR = f'{DRIVE_ROOT}/L24_output'
INPUT_DIR  = f'{DRIVE_ROOT}/inputs'          # the shared folder from L11/L12
for d in (OUTPUT_DIR, INPUT_DIR):
    os.makedirs(d, exist_ok=True)

# Models live on the runtime disk, not on Drive. Wiped when the runtime recycles
# (~2 min to re-download), and in exchange there is no OSError 95 from symlinks.
MODEL_CACHE = '/content/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME']        = MODEL_CACHE
os.environ['TORCH_HOME']     = MODEL_CACHE
os.environ['XDG_CACHE_HOME'] = MODEL_CACHE

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def vram(tag=''):
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM {used:5.2f} / {total:.1f} GB   {tag}')

print(torch.cuda.get_device_name(0)); vram('empty')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tesla T4
  VRAM  0.00 / 15.6 GB   empty


## 2 — Your audio half, from Friday

Three cells below are **yours to paste**, from your own L24 notebook — exactly as they ran
there. The contract cell between them is given again, unchanged, so paste around it.

| Paste into | From your L24 notebook |
|---|---|
| **2a** | section 2 — loading CLAP (`tagger`) and Whisper (`asr`) |
| **2b** | section 4 — `SCENES` and `CONF_FLOOR` (and anything else you defined there) |
| **2c** | section 5 — `tag_audio`, `transcribe`, `analyse_audio`, `run_audio` |

> **Did not finish Friday?** Then this is where today starts: finish those three cells first,
> here. The vision half needs `SCENES` to exist, and the join needs `analyse_audio` to work.

The check cell after them tells you whether your half survived the trip.

In [3]:
# 2a — L24 section 2: CLAP into `tagger` and Whisper into `asr`.
from transformers import pipeline
import whisper

tagger = pipeline('zero-shot-audio-classification',
                  model='laion/clap-htsat-unfused', device=0)
vram('after CLAP')

asr = whisper.load_model('small', download_root=MODEL_CACHE)
vram('both loaded')

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

  VRAM  0.62 / 15.6 GB   after CLAP
  VRAM  1.59 / 15.6 GB   both loaded


### The audio contract · **given, unchanged from Friday**

In [4]:
import librosa

# The rate each stage wants. Two different numbers in one pipeline.
CLAP_SR     = 48000     # measured: at any other rate CLAP fails intermittently
ASR_SR      = 16000     # Whisper
MAX_SECONDS = 60        # a 30-minute upload is the #1 cause of a stalled demo
MIN_SECONDS = 0.5


def list_inputs():
    """What is in the shared folder right now."""
    names = sorted(f for f in os.listdir(INPUT_DIR) if not f.startswith('.'))
    print(f'{INPUT_DIR}  ({len(names)} files)')
    for n in names:
        print(f'  {os.path.getsize(os.path.join(INPUT_DIR, n))/1e6:6.2f} MB  {n}')
    return names


def upload_inputs():
    """Pick files from your machine; they land in Drive and stay there."""
    from google.colab import files
    for fname, data in files.upload().items():
        with open(os.path.join(INPUT_DIR, fname), 'wb') as f:
            f.write(data)
        print(f'  saved {fname}')


def load_audio(name_or_path, sr=None):
    """A name in the inputs folder, or any path -> (wav float32 mono, sr)."""
    path = name_or_path
    if not os.path.isabs(path):
        path = os.path.join(INPUT_DIR, name_or_path)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{name_or_path} is not in {INPUT_DIR}. '
            'Run upload_inputs() and pick it, or copy it into that folder on Drive.')
    wav, got_sr = librosa.load(path, sr=sr, mono=True)
    return wav.astype('float32'), got_sr


def coerce_audio(wav, sr, target_sr):
    """Anything a user can hand us -> float32 mono at target_sr, or ValueError.

    This function is the contract. Call it ALWAYS, before a model sees anything.
    """
    if wav is None:
        raise ValueError('No audio received. Please upload a clip.')
    wav = np.asarray(wav)
    if wav.dtype.kind in 'iu':                       # gradio hands back int16
        wav = wav.astype('float32') / np.iinfo(wav.dtype).max
    wav = wav.astype('float32')
    if wav.ndim > 1:                                 # to mono, either layout
        wav = wav.mean(axis=0) if wav.shape[0] < wav.shape[1] else wav.mean(axis=1)
    if wav.size < sr * MIN_SECONDS:
        raise ValueError(f'Clip too short ({wav.size/sr:.2f} s). Use at least {MIN_SECONDS} s.')
    if wav.size > sr * MAX_SECONDS:
        wav = wav[:int(sr * MAX_SECONDS)]            # truncate, and the caller says so
    if np.abs(wav).max() < 1e-4:
        raise ValueError('That clip is silent. Whisper would invent words for it.')
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    return wav.astype('float32'), target_sr


print('contract loaded. CLAP_SR =', CLAP_SR, '| ASR_SR =', ASR_SR)

contract loaded. CLAP_SR = 48000 | ASR_SR = 16000


In [5]:
# 2b — PASTE your L24 section 4 here: SCENES, CONF_FLOOR, and anything else you defined.
SCENES = {
    'casa': [
        'a dog barking',
        'a baby crying',
        'a clock ticking',
        'a vacuum cleaner',
        'a person speaking',
    ],
    'exterior': [
        'rain falling',
        'birds chirping',
        'a chainsaw',
        'a car engine',
        'a person speaking',
    ],
}

# Cómo se nombra cada etiqueta en el reporte en español (si falta, se usa el inglés)
ES = {
    'a dog barking':     'un perro ladrando',
    'a baby crying':     'un bebé llorando',
    'a clock ticking':   'un reloj haciendo tic-tac',
    'a vacuum cleaner':  'una aspiradora',
    'a person speaking': 'una persona hablando',
    'rain falling':      'lluvia',
    'birds chirping':    'pájaros cantando',
    'a chainsaw':        'una motosierra',
    'a car engine':      'un motor de coche',
}

CONF_FLOOR = 0.8 #None   # se mide después de escribir tag_audio

In [6]:
# 2c — PASTE your L24 section 5 here: tag_audio, transcribe, analyse_audio, run_audio.
def tag_audio(wav, sr, labels):
    """(wav, sr) + a list of phrases -> [(label, score), ...] sorted, highest first."""
    assert sr == CLAP_SR, f'{sr} != {CLAP_SR} - coerce first'
    out = tagger(wav, candidate_labels=list(labels))
    return [(o['label'], float(o['score'])) for o in out]


def transcribe(wav, sr, language=None):
    """(wav, sr) -> text. Whisper wants float32 at 16 kHz."""
    assert sr == ASR_SR, f'{sr} != {ASR_SR} - coerce first'
    return asr.transcribe(wav, language=language, fp16=True)['text'].strip()


SPEECH_MIN_WORDS = 3   # Whisper siempre devuelve algo; con menos palabras no cuenta como voz


def analyse_audio(wav, sr, scene):
    if scene not in SCENES:
        raise ValueError(f'Escena desconocida: {scene!r}. Opciones: {", ".join(SCENES)}')

    # Dos tasas, dos coerciones, ambas desde el audio original
    w48, _ = coerce_audio(wav, sr, CLAP_SR)
    tags   = tag_audio(w48, CLAP_SR, SCENES[scene])

    w16, _ = coerce_audio(wav, sr, ASR_SR)
    text   = transcribe(w16, ASR_SR)
    speech = len(text.split()) >= SPEECH_MIN_WORDS

    top, conf = tags[0]
    if conf >= CONF_FLOOR:
        summary = f'Suena a {ES.get(top, top)}.'
    else:
        summary = 'No pude identificar este sonido con seguridad.'
    if speech:
        summary += f' Alguien dice: «{text}»'

    return {
        'modality': 'audio',
        'scene'   : scene,
        'tags'    : tags,                       # completo, aunque el resumen se niegue a adivinar
        'text'    : text if speech else None,
        'summary' : summary,
    }


def run_audio(x, sr=None, scene='casa'):
    try:
        if isinstance(x, str):                  # nombre en inputs/ o ruta
            wav, sr = load_audio(x)
        else:
            wav = x
            if sr is None:
                raise ValueError('Falta la tasa de muestreo: el audio siempre viaja como (wav, sr).')
        dur = max(np.asarray(wav).shape) / sr if wav is not None else 0
        t0  = time.time()
        a   = analyse_audio(wav, sr, scene)     # coerce_audio lanza ValueError si algo está mal
        dt  = time.time() - t0
    except ValueError as e:                     # culpa del usuario: se le dice qué arreglar
        return str(e), str(e)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        msg = 'La GPU se quedó sin memoria. Prueba con un clip más corto.'
        return msg, msg
    except Exception as e:                      # culpa nuestra: al log, no al usuario
        print(f'[run_audio] {type(e).__name__}: {e}')
        msg = 'Algo salió mal. Prueba con otro clip.'
        return msg, msg

    top3 = '\n'.join(f'  {label:22s} {score:.3f}' for label, score in a['tags'][:3])
    report = (f"{a['summary']}\n\n"
              f"escena : {a['scene']}\n"
              f"top-3  :\n{top3}\n"
              f"voz    : {a['text'] or '—'}\n"
              f"tiempo : {dt:.2f} s")
    if dur > MAX_SECONDS:
        report += f'\n\nNota: el clip duraba {dur:.0f} s y se recortó a {MAX_SECONDS} s.'
    return a['summary'], report

In [7]:
# ---------- check: did the audio half survive the trip? (given) ----------
assert tagger is not None and asr is not None, 'cell 2a: load both models'
assert SCENES, 'cell 2b: SCENES is empty'
assert isinstance(CONF_FLOOR, (int, float)), 'cell 2b: CONF_FLOOR must be a number you measured'

probe = [f for f in sorted(os.listdir(INPUT_DIR)) if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg'))]
wav, sr = load_audio(probe[0]) if probe else \
          ((0.2 * np.random.default_rng(SEED).standard_normal(CLAP_SR * 5)).astype('float32'), CLAP_SR)
a = analyse_audio(wav, sr, next(iter(SCENES)))
KEYS = {'modality', 'scene', 'tags', 'text', 'summary'}
assert set(a) >= KEYS, f'analyse_audio is missing keys: {KEYS - set(a)}'
assert a['modality'] == 'audio'
print('audio half OK on', probe[0] if probe else 'white noise', '->', a['summary'])
vram('audio half loaded')

audio half OK on birds.wav -> No pude identificar este sonido con seguridad.
  VRAM  1.60 / 15.6 GB   audio half loaded


## 3 — The image contract · **given**

Friday's contract was the sample rate. Today's is the one from L11: **anything a user can hand
the app becomes an RGB PIL image of a sane size**, or a `ValueError` that says what to fix.

Two things Gradio can hand you that are not what CLIP wants: a PNG with an alpha channel
(RGBA), and a black-and-white photo (mode `L`). Both have to become RGB before the model sees
them. The same function also stops a 6000-pixel phone photo from eating the GPU.

In [8]:
from PIL import Image

MAX_SIDE, MIN_SIDE = 1024, 32
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp')


def load_image(name_or_path):
    """A name in the inputs folder, or any path -> PIL image (not yet coerced)."""
    path = name_or_path
    if not os.path.isabs(path):
        path = os.path.join(INPUT_DIR, name_or_path)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{name_or_path} is not in {INPUT_DIR}. '
            'Run upload_inputs() and pick it, or copy it into that folder on Drive.')
    return Image.open(path)


def coerce_image(x):
    """Anything a user can hand us -> RGB PIL image, longest side <= MAX_SIDE, or ValueError.

    This is the L11 contract, unchanged. Call it ALWAYS, before a model sees anything.
    """
    if x is None:
        raise ValueError('No image received. Please upload one.')
    if isinstance(x, str):
        x = load_image(x)
    if isinstance(x, np.ndarray):
        x = Image.fromarray(x.astype(np.uint8))
    if not isinstance(x, Image.Image):
        raise ValueError('Unsupported input. Please upload an image file.')
    x = x.convert('RGB')                              # RGBA, L, P, CMYK -> RGB
    w, h = x.size
    if min(w, h) < MIN_SIDE:
        raise ValueError(f'Image too small ({w}x{h}). Use at least {MIN_SIDE} px.')
    if max(w, h) > MAX_SIDE:
        s = MAX_SIDE / max(w, h)
        x = x.resize((int(w * s), int(h * s)), Image.LANCZOS)
    return x


print('image contract loaded. MAX_SIDE =', MAX_SIDE)

image contract loaded. MAX_SIDE = 1024


## 4 — CLIP, and its floor · **you write them**

In L12 you called CLIP by hand: a processor, a model, `get_image_features`, a cosine
similarity. The `pipeline` does all of that in one line, and — this is the point — **its call
looks exactly like your CLAP call**:

```python
tagger(wav, candidate_labels=labels)          # Friday, audio
vision_tagger(img, candidate_labels=labels)   # today, images
```

The task is `zero-shot-image-classification`; the repo is `openai/clip-vit-base-patch32`.
Same output shape as CLAP: a list of dicts with `label` and `score`, highest first.

Three measured things worth knowing, on the images in the shared folder:

- **Your audio phrases work for images as they are.** The pipeline quietly wraps each one as
  `"This is a photo of {phrase}"` and hands your own label back. `'a dog barking'` found the
  dog photo at 0.98. You do not need a second vocabulary — that is the whole premise.
- **CLIP's scores are not on CLAP's scale.** A floor you measured for sound is not a floor for
  images. Measure this one separately, the same way: what does a hit score, what does a miss.
- **The confident mistake is back.** A photo of a bird, in a scene with no bird label, came back
  *"a person speaking"* at **0.96**. Exactly Friday's lesson: no floor catches that. The fix is
  still the vocabulary.

In [9]:
from transformers import pipeline

vision_tagger = pipeline('zero-shot-image-classification',
                         model='openai/clip-vit-base-patch32', device=0)

vram('after CLIP')        # expect ~0.6 GB more

# Provisional: se fija después de medir (celda de abajo)
IMAGE_CONF_FLOOR = 0.95

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  VRAM  2.20 / 15.6 GB   after CLIP


In [10]:
imgs = [f for f in sorted(os.listdir(INPUT_DIR)) if f.lower().endswith(IMAGE_EXTS)]

for scene, labels in SCENES.items():
    print(f'\n===== escena: {scene} =====')
    for name in imgs:
        out = vision_tagger(coerce_image(name), candidate_labels=labels)
        top = out[0]
        print(f"{top['score']:.2f}  {top['label']:20s}  {name[:45]}")


===== escena: casa =====
0.91  a vacuum cleaner      118534871-hombre-asiático-cortando-árboles-
0.99  a baby crying         bebe-llorando-emocion-poderosa-retrato-infant
1.00  a dog barking         dog-try-to-climb-tree-in-the-park-jack-russel
0.53  a clock ticking       free-photo-of-calle-de-la-ciudad-a-traves-de-
1.00  a clock ticking       hora.jpg
1.00  a vacuum cleaner      nueva-aspiradora-inalambrica-cvh-2-237956_120
0.91  a person speaking     pexels-photo-2393767.jpg
0.62  a vacuum cleaner      un-coche-deportivo-verde-se-sienta-en-la-pant
1.00  a person speaking     woman-standing-speaking-audience-conference-2

===== escena: exterior =====
1.00  a chainsaw            118534871-hombre-asiático-cortando-árboles-


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


0.89  a person speaking     bebe-llorando-emocion-poderosa-retrato-infant
0.70  a chainsaw            dog-try-to-climb-tree-in-the-park-jack-russel
1.00  rain falling          free-photo-of-calle-de-la-ciudad-a-traves-de-
0.69  a person speaking     hora.jpg
0.90  a chainsaw            nueva-aspiradora-inalambrica-cvh-2-237956_120
0.98  birds chirping        pexels-photo-2393767.jpg
1.00  a car engine          un-coche-deportivo-verde-se-sienta-en-la-pant
1.00  a person speaking     woman-standing-speaking-audience-conference-2


In [11]:
names = list_inputs()
print([f for f in names if f.lower().endswith(IMAGE_EXTS)])

/content/drive/MyDrive/TAE_IA_M6/inputs  (20 files)
    0.03 MB  118534871-hombre-asiático-cortando-árboles-usando-una-motosierra-eléctrica.jpg
    0.00 MB  L24_demo_audio
    0.03 MB  bebe-llorando-emocion-poderosa-retrato-infantil_1190912-23808.jpg
    0.44 MB  birds.wav
    0.44 MB  chainsaw.wav
    0.44 MB  clock_tick.wav
    0.44 MB  crying_baby.wav
    0.04 MB  dog-try-to-climb-tree-in-the-park-jack-russell-terrier-unleashed-in-backyard-dog-smile-free-photo.jpg
    0.44 MB  dog.wav
    0.07 MB  free-photo-of-calle-de-la-ciudad-a-traves-de-la-ventanilla-lateral-de-un-coche-empanada-y-mojada.jpg
    0.01 MB  hora.jpg
    0.01 MB  nueva-aspiradora-inalambrica-cvh-2-237956_1200x.jpg
    0.01 MB  pexels-photo-2393767.jpg
    0.44 MB  rain.wav
    0.04 MB  speech_en.mp3
    0.04 MB  speech_es.mp3
    0.11 MB  un-coche-deportivo-verde-se-sienta-en-la-pantalla-su-capó-abierto-para-revelar-motor-y-los-componentes-amarillos-están-visibles-346692194.jpg
    0.44 MB  vacuum_cleaner.wav


## 5 — The vision half · **you write it**

Three functions, deliberately the mirror image of Friday's. The signatures are given so the
join and the test cell can rely on them.

`analyse_image` returns **the same five keys** as `analyse_audio`. `modality` is `'image'`, and
`text` is `None` — there is nothing to transcribe in a photo.

In [12]:
def tag_image(img, labels):
    """PIL RGB image + a list of phrases -> [(label, score), ...] sorted, highest first."""
    out = vision_tagger(img, candidate_labels=list(labels))
    return sorted([(o['label'], float(o['score'])) for o in out], key=lambda t: -t[1])


def analyse_image(image, scene):
    """Anything a user can hand us + a scene name -> the same five-key dict as analyse_audio."""
    if scene not in SCENES:
        raise ValueError(f'Escena desconocida: {scene!r}. Opciones: {", ".join(SCENES)}')

    img  = coerce_image(image)                 # ValueError si la imagen no sirve
    tags = tag_image(img, SCENES[scene])

    top, conf = tags[0]
    if conf >= IMAGE_CONF_FLOOR:
        summary = f'Veo {ES.get(top, top)}.'
    else:
        summary = 'No pude identificar con seguridad lo que hay en la imagen.'

    return {
        'modality': 'image',
        'scene'   : scene,
        'tags'    : tags,                      # completo, aunque el resumen se niegue a adivinar
        'text'    : None,                      # en una foto no hay nada que transcribir
        'summary' : summary,
    }


def run_image(x, scene):
    """The image half's UI entry point. NEVER raises -> (summary, report)."""
    try:
        t0 = time.time()
        v  = analyse_image(x, scene)
        dt = time.time() - t0
    except (ValueError, FileNotFoundError) as e:     # culpa del usuario
        return str(e), str(e)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        msg = 'La GPU se quedó sin memoria. Prueba con otra imagen.'
        return msg, msg
    except Exception as e:                           # culpa nuestra: al log
        print(f'[run_image] {type(e).__name__}: {e}')
        msg = 'Algo salió mal. Prueba con otra imagen.'
        return msg, msg

    top3 = '\n'.join(f'  {label:22s} {score:.3f}' for label, score in v['tags'][:3])
    report = (f"{v['summary']}\n\n"
              f"escena : {v['scene']}\n"
              f"top-3  :\n{top3}\n"
              f"tiempo : {dt:.2f} s")
    return v['summary'], report

In [13]:
   print(run_image('hora.jpg', 'casa')[1])
   print(run_image('no_such_photo.jpg', 'casa')[1])

Veo un reloj haciendo tic-tac.

escena : casa
top-3  :
  a clock ticking        0.999
  a vacuum cleaner       0.000
  a person speaking      0.000
tiempo : 0.03 s
no_such_photo.jpg is not in /content/drive/MyDrive/TAE_IA_M6/inputs. Run upload_inputs() and pick it, or copy it into that folder on Drive.


## 6 — The join · **you write it**

This is the multimodal part, and it is short on purpose: both halves return the same shape
against the same vocabulary, so comparing them is comparing two lists.

The one decision that matters is **what to say when a side was not sure**. If the audio refused
to guess, "they disagree" would be a lie — the honest answer is "I cannot compare". That is why
`agree` has three values, and the test cell checks all three.

In [14]:
def join(a, v):
    """analyse_audio's dict + analyse_image's dict -> the verdict."""
    a_lab, a_conf = a['tags'][0]
    v_lab, v_conf = v['tags'][0]
    a_ok = a_conf >= CONF_FLOOR
    v_ok = v_conf >= IMAGE_CONF_FLOOR

    audio = a_lab if a_ok else None
    image = v_lab if v_ok else None

    if not (a_ok and v_ok):
        agree = None                                   # nada honesto que comparar
        if not a_ok and not v_ok:
            summary = 'Ni el sonido ni la imagen quedaron claros; no puedo compararlos.'
        elif not a_ok:
            summary = 'No pude identificar el sonido con seguridad, así que no puedo compararlo con la imagen.'
        else:
            summary = 'No pude identificar la imagen con seguridad, así que no puedo compararla con el sonido.'
    elif a_lab == v_lab:
        agree = True
        summary = f'Coinciden: el sonido y la imagen indican {ES.get(a_lab, a_lab)}.'
    else:
        agree = False
        summary = (f'No coinciden: oigo {ES.get(a_lab, a_lab)}, '
                   f'pero veo {ES.get(v_lab, v_lab)}.')

    return {'agree': agree, 'audio': audio, 'image': image, 'summary': summary}


def _try(fn, *args):
    """Corre una mitad de la app. -> (resultado, None) o (None, mensaje para el usuario)."""
    try:
        return fn(*args), None
    except (ValueError, FileNotFoundError) as e:       # culpa del usuario: se le dice qué arreglar
        return None, str(e)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, 'La GPU se quedó sin memoria. Prueba con entradas más pequeñas.'
    except Exception as e:                             # culpa nuestra: al log, no al usuario
        print(f'[run_both] {type(e).__name__}: {e}')
        return None, 'Algo salió mal. Prueba con otra entrada.'


def run_both(audio, image, scene):
    """The one function the app's button calls. NEVER raises -> (summary, report)."""
    if audio is None and image is None:
        msg = 'Sube un clip de audio, una imagen, o ambos.'
        return msg, msg

    a = v = None
    errs = []
    if audio is not None:
        # Gradio entrega (sr, wav): la tasa va PRIMERO
        a, err = _try(lambda: analyse_audio(audio[1], audio[0], scene))
        if err: errs.append(f'audio  : {err}')
    if image is not None:
        v, err = _try(analyse_image, image, scene)
        if err: errs.append(f'imagen : {err}')

    def top(d):
        label, score = d['tags'][0]
        return f'{label} ({score:.2f})'

    lines = [f'escena : {scene}']
    if a: lines.append(f"oído   : {a['summary']}  [{top(a)}]")
    if v: lines.append(f"visto  : {v['summary']}  [{top(v)}]")
    lines += errs

    if a and v:
        summary = join(a, v)['summary']
    elif a or v:
        falta = 'una imagen' if a else 'un clip de audio'
        summary = (a or v)['summary'] + f' Sube también {falta} para poder compararlos.'
    else:
        summary = ' '.join(e.split(': ', 1)[1] for e in errs)

    return summary, summary + '\n\n' + '\n'.join(lines)

In [15]:
w, s = load_audio('clock_tick.wav')
print(run_both((s, w), 'hora.jpg', 'casa')[1])          # debería coincidir (o decir que no está seguro)

w, s = load_audio('dog.wav')
print(run_both((s, w), 'hora.jpg', 'casa')[1])          # debería NO coincidir

print(run_both(None, 'hora.jpg', 'casa')[0])            # solo imagen: pide el audio
print(run_both(None, None, 'casa')[0])                  # nada: pide algo

Coinciden: el sonido y la imagen indican un reloj haciendo tic-tac.

escena : casa
oído   : Suena a un reloj haciendo tic-tac.  [a clock ticking (1.00)]
visto  : Veo un reloj haciendo tic-tac.  [a clock ticking (1.00)]
No coinciden: oigo un perro ladrando, pero veo un reloj haciendo tic-tac.

escena : casa
oído   : Suena a un perro ladrando.  [a dog barking (1.00)]
visto  : Veo un reloj haciendo tic-tac.  [a clock ticking (1.00)]
Veo un reloj haciendo tic-tac. Sube también un clip de audio para poder compararlos.
Sube un clip de audio, una imagen, o ambos.


## 7 — The test cell · **given. It has to pass.**

Four checks: hostile images never raise, the two halves return the same shape, the join tells
the truth in all three cases, and your images fire on your vocabulary.

In [16]:
scene = next(iter(SCENES))
fails = 0

# ---------- A. hostile images: none of these may raise ----------
print('--- hostile images ---')
cases = [
    ('None',          None),
    ('10x10 px',      Image.new('RGB', (10, 10), 'white')),
    ('6000x4000 px',  Image.new('RGB', (6000, 4000), 'steelblue')),
    ('grayscale L',   Image.new('L', (300, 300), 128)),
    ('RGBA',          Image.new('RGBA', (300, 300), (0, 128, 0, 100))),
    ('numpy uint8',   np.zeros((240, 320, 3), dtype=np.uint8)),
    ('missing file',  'no_such_photo.jpg'),
]
for tag, bad in cases:
    try:
        _s, msg = run_image(bad, scene)
        print(f'  {tag:14s} -> {(msg or _s or "").splitlines()[0][:60]}')
    except Exception as e:
        print(f'  {tag:14s} -> RAISED {type(e).__name__}: {e}')
        fails += 1

# ---------- B. the two halves speak the same shape ----------
print('\n--- shapes ---')
KEYS = {'modality', 'scene', 'tags', 'text', 'summary'}
v = analyse_image(Image.new('RGB', (256, 256), 'steelblue'), scene)
assert set(v) >= KEYS, f'analyse_image is missing keys: {KEYS - set(v)}'
assert v['modality'] == 'image' and v['text'] is None
assert {t for t, _ in v['tags']} == set(SCENES[scene]), 'image tags must use the SAME vocabulary'
print('  analyse_image returns the five keys, on the same vocabulary')

# ---------- C. the join tells the truth ----------
print('\n--- join ---')
L1, L2 = SCENES[scene][0], SCENES[scene][1]
def fake(modality, label, score):
    rest = [l for l in SCENES[scene] if l != label]
    tags = [(label, score)] + [(l, (1 - score) / len(rest)) for l in rest]
    return {'modality': modality, 'scene': scene, 'text': None, 'summary': '',
            'tags': sorted(tags, key=lambda t: -t[1])}          # the contract: highest first
for name, a_, v_, want in [
        ('same label',      fake('audio', L1, 0.99), fake('image', L1, 0.99), True),
        ('different label', fake('audio', L1, 0.99), fake('image', L2, 0.99), False),
        ('audio unsure',    fake('audio', L1, 0.01), fake('image', L1, 0.99), None),
        ('image unsure',    fake('audio', L1, 0.99), fake('image', L1, 0.01), None)]:
    j = join(a_, v_)
    ok = j.get('agree') is want and {'agree', 'audio', 'image', 'summary'} <= set(j)
    fails += not ok
    print(f'  {name:16s} -> agree={j.get("agree")!s:5s} {"ok" if ok else "*** expected " + str(want)}  {j.get("summary", "")[:50]}')

# ---------- D. your images against your vocabulary ----------
print('\n--- your images ---')
imgs = [f for f in sorted(os.listdir(INPUT_DIR)) if f.lower().endswith(IMAGE_EXTS)]
if not imgs:
    print('  (no images in the folder - run upload_inputs())')
for name in imgs[:6]:
    s, _ = run_image(name, scene)
    print(f'  {name}: {s}')

vram('peak')
print(f"\npeak {torch.cuda.max_memory_allocated()/1e9:.2f} GB of "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print('\nfailures:', fails, '(must be 0)')

--- hostile images ---
  None           -> No image received. Please upload one.
  10x10 px       -> Image too small (10x10). Use at least 32 px.
  6000x4000 px   -> No pude identificar con seguridad lo que hay en la imagen.
  grayscale L    -> No pude identificar con seguridad lo que hay en la imagen.
  RGBA           -> No pude identificar con seguridad lo que hay en la imagen.
  numpy uint8    -> No pude identificar con seguridad lo que hay en la imagen.
  missing file   -> no_such_photo.jpg is not in /content/drive/MyDrive/TAE_IA_M6

--- shapes ---
  analyse_image returns the five keys, on the same vocabulary

--- join ---
  same label       -> agree=True  ok  Coinciden: el sonido y la imagen indican un perro 
  different label  -> agree=False ok  No coinciden: oigo un perro ladrando, pero veo un 
  audio unsure     -> agree=None  ok  No pude identificar el sonido con seguridad, así q
  image unsure     -> agree=None  ok  No pude identificar la imagen con seguridad, así q

--- your

## 8 — The app · **you write it**

One interface for the whole thing. Everything you need you have used before: `Blocks`, `Row`,
`Column`, `Audio`, `Image`, `Dropdown`, `Button`, `Textbox`, `.click()`, `.queue()`, `.launch()`.

The two contracts meet here for the first time, so write them down before you wire anything:

```python
gr.Audio(type='numpy')   # the callback receives (sr, wav) — rate FIRST
gr.Image(type='pil')     # the callback receives a PIL image, or None
```

Minimum requirements:

- `gr.Audio(type='numpy')` **and** `gr.Image(type='pil')` as inputs
- a `gr.Dropdown` for the scene, fed from `SCENES`
- **one** button that runs `run_both`
- a `gr.Textbox` with the report
- `.queue()` before `.launch(share=True)`

Tabs (`gr.Tab`) are allowed — you saw them in the demo — but one screen is simpler, and the
point of this app is seeing both answers side by side.

In [17]:
import gradio as gr
print('gradio', gr.__version__)

try:
    demo.close()        # re-running this cell: stop the previous app first
except NameError:
    pass


def on_click(audio, image, scene):
    return run_both(audio, image, scene)[1]     # el reporte ya incluye el veredicto


with gr.Blocks(title='¿Qué suena aquí?') as demo:
    gr.Markdown('# ¿Qué suena aquí?\n'
                'Sube un clip de audio y una foto: la app dice qué oye, qué ve '
                'y si ambos coinciden.')
    scene_dd = gr.Dropdown(list(SCENES), value=next(iter(SCENES)), label='Escena')
    with gr.Row():
        aud_in = gr.Audio(type='numpy', label='Tu clip')    # (sr, wav): la tasa primero
        img_in = gr.Image(type='pil', label='Tu foto')      # PIL o None
    go  = gr.Button('¿Coinciden?', variant='primary')
    out = gr.Textbox(label='Reporte', lines=10)
    go.click(on_click, [aud_in, img_in, scene_dd], out)     # entradas por ORDEN

demo.queue().launch(share=True, quiet=True)

gradio 6.26.0
* Running on public URL: https://79b3657ef004cbc04c.gradio.live


In [19]:
# Stop the app when you are done with it.
demo.close()

Closing server running on port: 7860


---

## If you get stuck

**Test each half alone before the join.** `run_audio('dog.wav', scene=...)` and
`run_image('dog.jpg', scene)` should each make sense on their own. If one of them does not,
`join` cannot fix it.

**If the image half names everything the same thing**, print `tag_image(...)` for two very
different photos. If the scores barely move, you are probably passing the same image every
time — or an image that never went through `coerce_image`.

**If the join says "no coinciden" on a dog barking and a dog photo**, print both `tags[0]`.
Labels are compared as strings: `'a dog barking'` and `'a dog barking '` are different labels.

## The showcase

Show what runs, finished or not:

1. Pick a scene, and show one pair that **agrees** — a sound and a photo of the same thing.
2. Show one pair that **disagrees**, and say whether the app was right to say so.
3. Show one case where the app **refused** — and one where it was **confidently wrong**. Say
   which label was missing from your vocabulary.

Number 3 is the one that shows you understood the app. Anyone can demo a success.

## Before you leave

- [ ] The test cell reports **0** failures
- [ ] `IMAGE_CONF_FLOOR` is a number you **measured**, separately from `CONF_FLOOR`
- [ ] The app takes a clip **and** a photo and says whether they agree
- [ ] **Restart the runtime and run everything again**

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L25*